In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# Day 1: Load Datasets

customer_df = pd.read_csv("customer_data.csv")
sales_df = pd.read_csv("sales_data.csv")

customer_df.columns = customer_df.columns.str.strip()
sales_df.columns = sales_df.columns.str.strip()

print("Customer Columns:")
print(customer_df.columns.tolist())

print("\nSales Columns:")
print(sales_df.columns.tolist())

# Day 2: Data Cleaning

num_cols = customer_df.select_dtypes(include='number').columns
customer_df[num_cols] = customer_df[num_cols].fillna(
    customer_df[num_cols].median()
)

cat_cols = customer_df.select_dtypes(include='object').columns
customer_df[cat_cols] = customer_df[cat_cols].fillna("Unknown")

num_cols_sales = sales_df.select_dtypes(include='number').columns
sales_df[num_cols_sales] = sales_df[num_cols_sales].fillna(
    sales_df[num_cols_sales].median()
)

cat_cols_sales = sales_df.select_dtypes(include='object').columns
sales_df[cat_cols_sales] = sales_df[cat_cols_sales].fillna("Unknown")

sales_df['Date'] = pd.to_datetime(
    sales_df['Date'],
    dayfirst=True,
    errors='coerce'
)

sales_df = sales_df.dropna(subset=['Date'])

sales_df['Total_Sales'] = pd.to_numeric(
    sales_df['Total_Sales'],
    errors='coerce'
)

sales_df = sales_df.dropna(subset=['Total_Sales'])

print("\nData Cleaning Completed")

# Day 3: Customer Analysis

print("\n===== CUSTOMER ANALYSIS =====")

print("\nCustomer Lifetime Value:")
print(customer_df[['CustomerID', 'TotalCharges']].head())

churn_rate = customer_df['Churn'].mean() * 100
print(f"\nChurn Rate: {churn_rate:.2f}%")

print("\nContract Distribution:")
print(customer_df['Contract'].value_counts())

print("\nPayment Method Distribution:")
print(customer_df['PaymentMethod'].value_counts())

# Day 4: Sales Analysis

print("\n===== SALES ANALYSIS =====")

monthly_sales = sales_df.groupby(
    sales_df['Date'].dt.to_period('M')
)['Total_Sales'].sum()

print("\nMonthly Sales:")
print(monthly_sales)

best_products = sales_df.groupby('Product')['Quantity'] \
                        .sum() \
                        .sort_values(ascending=False)

print("\nBest Selling Products:")
print(best_products)

regional_sales = sales_df.groupby('Region')['Total_Sales'] \
                         .sum() \
                         .sort_values(ascending=False)

print("\nRegional Sales:")
print(regional_sales)

# Day 5: Advanced Analysis

pivot_table = pd.pivot_table(
    sales_df,
    values='Total_Sales',
    index='Region',
    columns='Product',
    aggfunc='sum',
    fill_value=0
)

print("\nPivot Table:")
print(pivot_table)

top_products = sales_df.groupby('Product')['Total_Sales'] \
                       .sum() \
                       .sort_values(ascending=False)

print("\nTop Products by Revenue:")
print(top_products)

regional_sales.to_csv("regional_sales.csv")
top_products.to_csv("top_products.csv")

# Day 6: Dashboard Creation

os.makedirs("visualizations", exist_ok=True)

if len(monthly_sales) > 0:
    plt.figure(figsize=(8, 5))
    monthly_sales.plot(marker='o')
    plt.title("Monthly Sales Trend")
    plt.xlabel("Month")
    plt.ylabel("Total Sales")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig("visualizations/monthly_sales_trend.png")
    plt.close()

if len(top_products) > 0:
    plt.figure(figsize=(8, 5))
    top_products.head(10).plot(kind='bar')
    plt.title("Top Products by Revenue")
    plt.xlabel("Product")
    plt.ylabel("Revenue")
    plt.tight_layout()
    plt.savefig("visualizations/top_products_revenue.png")
    plt.close()

if len(regional_sales) > 0:
    plt.figure(figsize=(7, 7))
    regional_sales.plot(kind='pie', autopct='%1.1f%%')
    plt.title("Regional Sales Distribution")
    plt.ylabel("")
    plt.tight_layout()
    plt.savefig("visualizations/regional_sales_distribution.png")
    plt.close()

contract_counts = customer_df['Contract'].value_counts()

if len(contract_counts) > 0:
    plt.figure(figsize=(7, 7))
    contract_counts.plot(kind='pie', autopct='%1.1f%%')
    plt.title("Contract Distribution")
    plt.ylabel("")
    plt.tight_layout()
    plt.savefig("visualizations/contract_distribution.png")
    plt.close()

churn_counts = customer_df['Churn'].value_counts()

if len(churn_counts) > 0:
    plt.figure(figsize=(8, 5))
    churn_counts.plot(kind='bar')
    plt.title("Churn Distribution")
    plt.xlabel("Churn (0 = No, 1 = Yes)")
    plt.ylabel("Number of Customers")
    plt.tight_layout()
    plt.savefig("visualizations/churn_distribution.png")
    plt.close()

print("\nDashboard visualizations saved successfully!")
print("Folder: visualizations/")

# Day 7: Report & Insights

print("\nAnalysis Complete!")

Customer Columns:
['CustomerID', 'Tenure', 'MonthlyCharges', 'TotalCharges', 'Contract', 'PaymentMethod', 'PaperlessBilling', 'SeniorCitizen', 'Churn']

Sales Columns:
['Date', 'Product', 'Quantity', 'Price', 'Customer_ID', 'Region', 'Total_Sales']

Data Cleaning Completed

===== CUSTOMER ANALYSIS =====

Customer Lifetime Value:
  CustomerID  TotalCharges
0     C00001          1540
1     C00002          1753
2     C00003          1455
3     C00004          7150
4     C00005          1023

Churn Rate: 10.60%

Contract Distribution:
Contract
One year          186
Month-to-month    170
Two year          144
Name: count, dtype: int64

Payment Method Distribution:
PaymentMethod
Credit Card         178
Electronic Check    163
Bank Transfer       159
Name: count, dtype: int64

===== SALES ANALYSIS =====

Monthly Sales:
Date
2024-01    630943
2024-02    537878
2024-03    458714
2024-04    283098
2024-05    979464
2024-06    606520
2024-07    742872
2024-08    224384
2024-09    330916
2024-10  